# 简单 Monte Carlo 积分

> 对应代码：`MathStatsCode/code_in_notes/Chap.MCMC/MonteCarlo_simple.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：用均匀分布抽样的 Monte Carlo 方法计算二元函数 h(x,y) 在 [-1,1]×[-1,1] 上的积分。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.MCMC`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.MCMC")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**简单的Monte Carlo方法示例**

`clear all` 清空全部数据与数据框，`set more off` 关闭分页，`set seed 8855` 固定随机数种子，保证结果可复现。


In [ ]:
%%stata
clear all
set more off
set seed 8855


**设定参数**

`local M = 10000`：用局部宏设定模拟次数 M=10000。**Monte Carlo 积分的基本思想**：把积分写成“函数值×概率密度”的期望形式，再用大量随机样本的平均值来近似——模拟次数越多，精度越高（标准误按 1/√M 递减）。


In [ ]:
%%stata
local M = 10000


**抽样：从[-1,1]x[-1,1]的均匀分布中抽样**

在 [-1,1]×[-1,1] 的矩形区域内**均匀抽样**：`set obs \`M'` 建立 M 个观测，`gen x1 = runiform()*2-1` 把 [0,1] 的均匀随机数线性变换到 [-1,1]（同理 x2）。这一步对应积分公式中被积函数“定义域为矩形”的情形。


In [ ]:
%%stata
set obs `M'
gen x1 = runiform()*2-1
gen x2 = runiform()*2-1


**计算h(x)**

计算被积函数 h(x1,x2)=0.5·exp(-90(x1-0.5)²-45(x2+0.1)²) 并做 Monte Carlo 积分：
- `gen h=...` 计算每个样本点的函数值；
- `qui: su h` 求样本均值与标准差——积分估计等于“区域面积4 × 平均函数值”（`local integral=4*r(mean)`），因为均匀分布在矩形上的密度为1/4；
- 标准误按 4·sd(h)/√M 计算，并给出 95% 置信区间；
- `count if h>0.01` 统计函数值显著大于0的样本比例（约5%），显示**大部分抽样都“浪费”在函数值几乎为0的区域**——这正是下一节引入“重要性抽样”的动机。


In [ ]:
%%stata
gen h = 0.5*exp(-90*(x1-0.5)^2 - 45*(x2+0.1)^2)
qui: su h
local integral = 4*r(mean)
local se = 4*r(sd)/sqrt(`M')
di "Integral = " `integral'
di "s.e. of Integral = " `se'
di "95% C.I.: " `integral'-1.96*`se' " ~ " `integral'+1.96*`se'
qui: count if h>0.01
di "Ratio of >0.01 samples: " r(N)/`M'
